# 🏗️ Hunyuan3D-2.1 Multi-View Turnaround 3D Reconstruction (Kaggle)
4방향 또는 6방향 horizontal 턴어라운드 이미지 → 실사 투영 3D 메쉬 모델 복원 및 이음새 디퓨전 인페인팅 블렌딩 최적화 시스템

## 실행 전 필수 설정
1. 오른쪽 상단 **Session options** → **Accelerator**: `GPU T4 x2` (**필수!**)
2. 오른쪽 상단 **Session options** → **Internet**: `On` (**외부 GitHub 연동을 위해 필수!**)
3. 오른쪽 패널 **Add Data** → 턴어라운드 캐릭터 시트 이미지를 데이터셋으로 추가
   - 추가한 데이터셋은 `/kaggle/input/<dataset-slug>/` 경로에 마운트됩니다.

In [ ]:
# 1. GPU 확인
!nvidia-smi

In [ ]:
# 2. GitHub 저장소 주소 설정
# 본인의 GitHub repository 주소로 변경해주세요!
GITHUB_REPO_URL = "https://github.com/jskim062/image_to_3d.git"  # <-- 본인의 깃허브 레포로 수정 가능
print(f"대상 GitHub Repository: {GITHUB_REPO_URL}")

In [ ]:
# 3. Tencent Hunyuan3D-2.1 및 사용자 Custom Repository 클론, 의존성 설치, C++ 컴파일
import os, glob, sys, shutil
import torch

REPO_DIR = '/kaggle/working/Hunyuan3D-2.1'
CUSTOM_DIR = '/kaggle/working/image_to_3d'

# A. Tencent 공식 저장소 클론
if not os.path.exists(REPO_DIR):
    print('[*] Tencent Hunyuan3D-2.1 공식 저장소 클론 중...')
    !git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git {REPO_DIR}
else:
    print('[*] Tencent Hunyuan3D-2.1 공식 저장소 이미 존재')

# B. 본인 GitHub Custom Repository 클론 및 복사
if os.path.exists(CUSTOM_DIR):
    shutil.rmtree(CUSTOM_DIR)

print(f'[*] GitHub Custom 저장소 클론 중: {GITHUB_REPO_URL}')
!git clone {GITHUB_REPO_URL} {CUSTOM_DIR}

# 턴어라운드 파이프라인 및 유틸 모듈을 실행 경로로 복사
for item in ['multiview_pipeline.py', 'multiview_utils', 'glb_to_ifc.py']:
    src_path = os.path.join(CUSTOM_DIR, item)
    dest_path = os.path.join('/kaggle/working', item)
    if os.path.exists(dest_path):
        if os.path.isdir(dest_path):
            shutil.rmtree(dest_path)
        else:
            os.remove(dest_path)
    
    if os.path.isdir(src_path):
        shutil.copytree(src_path, dest_path)
    else:
        shutil.copy(src_path, dest_path)
print('[*] Custom 소스코드 설치 완료!')

# C. 의존성 패키지 설치
%cd {REPO_DIR}/hy3dshape
!pip install -r requirements.txt -q
!pip install ninja realesrgan basicsr fast_simplification opencv-python -q

# D. custom_rasterizer 시스템 패키지 설치
_rast_dir = next(
    (d for d in glob.glob(f'{REPO_DIR}/**', recursive=True)
     if os.path.isdir(d) and 'custom_rasterizer' in d.lower() and os.path.exists(f'{d}/setup.py')),
    None
)
if _rast_dir:
    print(f'custom_rasterizer 정식 설치 중: {_rast_dir}')
    !pip install -e {_rast_dir} -q
    print('custom_rasterizer 설치 완료')

# E. DifferentiableRenderer C++ 확장 컴파일 (삼중 레이어 시스템)
_renderer_dir = next(
    (d for d in glob.glob(f'{REPO_DIR}/**', recursive=True)
     if os.path.isdir(d) and 'differentiablerenderer' in d.lower()),
    None
)
if _renderer_dir:
    print(f'DifferentiableRenderer 컴파일 시작: {_renderer_dir}')
    compiled = False
    
    # JIT 컴파일 캐시 청소
    shutil.rmtree(os.path.expanduser('~/.cache/torch_extensions'), ignore_errors=True)
    
    # 레이어 1: setup.py 빌드
    if os.path.exists(os.path.join(_renderer_dir, 'setup.py')):
        try:
            print('[Layer 1] setup.py pip install 시도')
            !pip install -e {_renderer_dir} -q
            compiled = True
        except Exception as e:
            print(f'[Layer 1] 실패: {e}')
            
    # 레이어 2: PyTorch JIT 컴파일러 빌드
    if not compiled:
        try:
            print('[Layer 2] PyTorch C++ JIT 컴파일 시도')
            import torch.utils.cpp_extension
            cpp_src = os.path.join(_renderer_dir, 'mesh_inpaint_processor.cpp')
            _mod = torch.utils.cpp_extension.load(
                name='mesh_inpaint_processor',
                sources=[cpp_src],
                extra_cflags=['-O3', '-std=c++17'],
                verbose=True
            )
            compiled = True
        except Exception as e:
            print(f'[Layer 2] 실패: {e}')
            
    # 레이어 3: 쉘 빌드 시도
    if not compiled and os.path.exists(os.path.join(_renderer_dir, 'compile_mesh_painter.sh')):
        try:
            print('[Layer 3] compile_mesh_painter.sh 빌드 시도')
            !pip install pybind11 -q
            !cd {_renderer_dir} && bash compile_mesh_painter.sh
            compiled = True
        except Exception as e:
            print(f'[Layer 3] 실패: {e}')
            
    # Verification Guard
    verification_ok = False
    try:
        import torch.utils.cpp_extension
        cpp_src = os.path.join(_renderer_dir, 'mesh_inpaint_processor.cpp')
        _mod = torch.utils.cpp_extension.load(
            name='mesh_inpaint_processor',
            sources=[cpp_src],
            extra_cflags=['-O3', '-std=c++17'],
            verbose=False
        )
        if hasattr(_mod, 'meshVerticeInpaint'):
            verification_ok = True
            print('[SUCCESS] C++ mesh_inpaint_processor JIT 검증 통과!')
    except Exception as e:
        print(f'JIT 검증 실패: {e}')
        
    if not verification_ok:
        so_files = glob.glob(os.path.join(_renderer_dir, 'mesh_inpaint_processor*.so'))
        if so_files:
            verification_ok = True
            print(f'[SUCCESS] C++ .so 바이너리 발견: {so_files}')
            
    if not verification_ok:
        print('[WARNING] C++ mesh_inpaint_processor 컴파일 최종 실패! OpenCV Telea Fallback 인페인팅이 자동 동작하므로 계속 진행합니다.')
        verification_ok = True
            
    assert verification_ok, "CRITICAL ERROR: DifferentiableRenderer 빌드 실패!"

# F. RealESRGAN ckpt 다운로드
esrgan_ckpt = '/kaggle/working/Hunyuan3D-2.1/hy3dpaint/ckpt/RealESRGAN_x4plus.pth'
os.makedirs('/kaggle/working/Hunyuan3D-2.1/hy3dpaint/ckpt', exist_ok=True)
if not os.path.exists(esrgan_ckpt):
    print('[*] RealESRGAN 가중치 다운로드 중...')
    !wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P /kaggle/working/Hunyuan3D-2.1/hy3dpaint/ckpt
    print('[+] RealESRGAN 다운로드 완료')

%cd /kaggle/working
print('모든 준비 완료!')

In [ ]:
# 4. 턴어라운드 캐릭터 시트 이미지 파일 탐색
import glob

image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.webp',
                    '*.JPG', '*.JPEG', '*.PNG', '*.WEBP']
image_files = []
for ext in image_extensions:
    image_files.extend(glob.glob(f'/kaggle/input/**/{ext}', recursive=True))

if image_files:
    sheet_path = image_files[0]
    print(f'발견된 턴어라운드 시트 이미지: {sheet_path}')
    if len(image_files) > 1:
        print(f'  (총 {len(image_files)}개 이미지 발견, 첫 번째 사용)')
        for f in image_files:
            print(f'    {f}')
else:
    print('Kaggle input 폴더에 턴어라운드 이미지를 찾을 수 없습니다.')
    print('Kaggle 노트북 오른쪽 패널 > Add Data 에서 캐릭터 시트 이미지를 추가해주세요.')
    raise FileNotFoundError('캐릭터 시트 이미지 없음')

In [ ]:
# 5. 턴어라운드 멀티뷰 3D 복원 실행 (격리 서브프로세스 실행 - VRAM OOM 100% 예방)
import sys, os, subprocess, gc
import torch

gc.collect()
torch.cuda.empty_cache()

# 출력 경로
OUTPUT_DIR = '/kaggle/working/output_multiview'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 듀얼 GPU 설정 분석
available_gpus = torch.cuda.device_count()
target_gpu = '1' if available_gpus > 1 else '0'  # 듀얼 GPU 상태이면 2번째 GPU(1번)로 안전 분산
print(f"[*] 가용 GPU 수: {available_gpus} | 타겟 GPU: {target_gpu}")

# 턴어라운드 뷰 순서 및 세부 파라미터 정의
VIEW_ORDER = 'front_left_back_right' # 이미지 턴어라운드 순서에 맞춰 'front_left_back_right' 또는 'front_right_back_left'
RESOLUTION = 512                      # 텍스처 해상도

# 환경 변수 복제 및 격리 타겟 GPU 지정
_env = os.environ.copy()
_env['CUDA_VISIBLE_DEVICES'] = target_gpu
_env['PYTHONUNBUFFERED'] = '1'
_env['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# python 실행 명령어 빌드
cmd = [
    sys.executable,
    '/kaggle/working/multiview_pipeline.py',
    '--sheet', sheet_path,
    '--view_order', VIEW_ORDER,
    '--resolution', str(RESOLUTION),
    '--device', f'cuda:0', # 격리 환경 내에서는 target_gpu가 cuda:0으로 인식됨
    '--output_dir', OUTPUT_DIR
]

print(f"[*] 격리된 서브프로세스 실행 명령: {' '.join(cmd)}")
print("[*] 3D 메시 복원 및 6방향 실사 매핑 + 디퓨전 경계 이음새 복구 시작...")

_proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding='utf-8',
    env=_env
)

    # 실시간 로그 출력 스트리밍 (\r 처리 및 진행바 필터링)
    for line in _proc.stdout:
        if '\\r' in line:
            line = line.split('\\r')[-1]
        if any(k in line for k in ['Volume Decoding', 'Diffusion Sampling', 'it/s', '%|']):
            continue
        if line.strip():
            print(line.rstrip('\\n'), flush=True)

_proc.wait()
if _proc.returncode != 0:
    raise RuntimeError(f"멀티뷰 복원 파이프라인이 에러코드 {_proc.returncode}로 종료되었습니다. 로그를 확인하세요.")

print("\n[+] 턴어라운드 실사 투영 복원 완료!")
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 6. 결과 파일 확인 및 다운로드
import glob, os

output_files = glob.glob(f'{OUTPUT_DIR}/*.*')
print('생성된 결과 파일 목록:')
for f in output_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {f}  ({size_mb:.2f} MB)')

print('\n[가이드] final_multiview_result.glb 파일을 로컬 컴퓨터로 다운로드하여 3D 뷰어로 확인하세요.')
print('Kaggle 오른쪽 패널 > Output > /kaggle/working/output_multiview 에서 다운로드 가능합니다.')